# LWFA Data Analysis Example

This notebook demonstrates how to load, process, and visualize PIC simulation data from laser wakefield acceleration simulations.

In [1]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

# Add src to path
sys.path.insert(0, str(Path.cwd().parent.parent / 'src'))

from pipeline.loaders import load_data
from pipeline.processors import DataProcessor
from pipeline.visualizers import Visualizer

%matplotlib inline
plt.rcParams['figure.dpi'] = 100

## 1. Load Data

In [2]:
# Path to data file
data_file = Path.cwd().parent / 'data' / 'example_lwfa.h5'

# Load data
loader = load_data(data_file)

# Check available fields and species
print("Available fields:", loader.list_fields())
print("Available species:", loader.list_species())

# Get metadata
metadata = loader.get_metadata()
print("\nMetadata:", metadata)

Available fields: ['Ex', 'Ey', 'Rho_electron']
Available species: ['electron']

Metadata: {'iteration': 1000, 'lambda_p': 3.345122020822132e-05, 'laser_a0': 2.0, 'plasma_density': 1e+24, 'time': 5e-14}


## 2. Load Fields

In [3]:
# Load electron density
density_data = loader.load_field('Rho_electron')
n_e = density_data['data']
x = density_data['grid_x']
y = density_data['grid_y']

# Load electric fields
E_y = loader.load_field('Ey')['data']
E_x = loader.load_field('Ex')['data']

print(f"Grid shape: {n_e.shape}")
print(f"X range: {x[0]*1e6:.2f} to {x[-1]*1e6:.2f} μm")
print(f"Y range: {y[0]*1e6:.2f} to {y[-1]*1e6:.2f} μm")

Grid shape: (200, 400)
X range: 0.00 to 40.00 μm
Y range: -10.00 to 10.00 μm


## 3. Load Particles

In [4]:
# Load electron particles
particles = loader.load_particles('electron')

print(f"Number of particles: {len(particles['x'])}")
print(f"Particle data keys: {list(particles.keys())}")

Number of particles: 1501
Particle data keys: ['x', 'y', 'px', 'py', 'pz', 'weight']


## 4. Process Data

In [5]:
processor = DataProcessor()

# Compute particle energies
energy_MeV = processor.compute_particle_energy(
    particles['px'], particles['py'], particles['pz']
)

print(f"Mean energy: {np.mean(energy_MeV):.2f} MeV")
print(f"Max energy: {np.max(energy_MeV):.2f} MeV")
print(f"Energy spread: {np.std(energy_MeV):.2f} MeV")

# Total charge
charge_pC = processor.compute_charge(particles['weight'])
print(f"Total charge: {charge_pC:.2f} pC")

Mean energy: 98.82 MeV
Max energy: 129.28 MeV
Energy spread: 9.66 MeV
Total charge: 0.00 pC


## 5. Visualizations

In [6]:
# Initialize visualizer
visualizer = Visualizer(output_dir='../../output/notebook')

### Density with E-field Overlay

In [7]:
visualizer.plot_density_with_field_overlay(
    x=x, y=y, density=n_e, field=E_y,
    output_name='notebook_density_ey',
    title='Electron Density with Transverse E-Field',
    save_png=True, save_html=True,
    smooth_density=0.5, smooth_field=0.5
)

Saved PNG: ../../output/notebook/notebook_density_ey.png
Saved HTML: ../../output/notebook/notebook_density_ey.html


### Energy Spectrum

In [8]:
# Compute spectrum
energy_bins, spectrum = processor.compute_energy_spectrum(
    particles['px'], particles['py'], particles['pz'],
    weight=particles['weight'], bins=50
)

# Plot
visualizer.plot_energy_spectrum(
    energy=energy_bins, spectrum=spectrum,
    output_name='notebook_spectrum',
    title=f'Energy Spectrum (Q = {charge_pC:.2f} pC)',
    save_png=True, save_html=True
)

Saved PNG: ../../output/notebook/notebook_spectrum.png
Saved HTML: ../../output/notebook/notebook_spectrum.html


### Phase Space Plots

In [9]:
# x-px phase space
visualizer.plot_phase_space(
    x=particles['x'], px=particles['px'],
    weight=particles['weight'],
    output_name='notebook_phase_x_px',
    title='Phase Space: x-px',
    save_png=True, save_html=True
)

Saved PNG: ../../output/notebook/notebook_phase_x_px.png
Saved HTML: ../../output/notebook/notebook_phase_x_px.html


### Particle Distribution

In [10]:
visualizer.plot_particle_distribution(
    x=particles['x'], y=particles['y'],
    color_by=energy_MeV,
    output_name='notebook_particles',
    title='Particle Spatial Distribution',
    color_label='Energy [MeV]',
    save_png=True, save_html=True,
    alpha=0.6, marker_size=2.0
)

Saved PNG: ../../output/notebook/notebook_particles.png
Saved HTML: ../../output/notebook/notebook_particles.html


## Summary

This notebook demonstrated:
- Loading PIC simulation data (HDF5 format)
- Processing fields and particles
- Computing energy spectra and beam properties
- Creating publication-quality visualizations

All outputs are saved in the `output/notebook/` directory as both PNG and interactive HTML files.